# Assignment 1 — Pandas Practice Project (QuickMart)

Build a small **sales analytics report** step by step from the QuickMart raw
data. Each **TASK** is a few small cells — run them one at a time and discuss
the output as we go.

Dataset: `datasets/new-raw/` (`orders`, `order_items`, `products`, `customers`).
The raw files are **dirty** (duplicates, blank/invalid values, `qty=0`,
wrong subtotals), so you clean them first — then analyse.


In [1]:
import os
import pandas as pd

# resolve datasets/new-raw no matter where Jupyter is launched from
DATASETS = next(
    os.path.join(p, "datasets")
    for p in [".", "..", "../..", "../../.."]
    if os.path.isdir(os.path.join(p, "datasets"))
)
RAW = os.path.join(DATASETS, "new-raw")
print("raw data at", RAW)

raw data at ../../datasets/new-raw


### TASK 1 — Load the four raw CSVs.

In [2]:
products  = pd.read_csv(os.path.join(RAW, "products.csv"))
order_items = pd.read_csv(os.path.join(RAW, "order_items.csv"))
orders    = pd.read_csv(os.path.join(RAW, "orders.csv"), parse_dates=["order_date"])
customers = pd.read_csv(os.path.join(RAW, "customers.csv"))

In [3]:
for name, d in [("products", products), ("order_items", order_items),
                ("orders", orders), ("customers", customers)]:
    print(f"{name:12s}: {d.shape}")

products    : (67, 5)
order_items : (9382, 6)
orders      : (3235, 5)
customers   : (1090, 5)


### TASK 2 — Clean each table.

A warehouse is schema-on-write: bad rows shouldn't make it into the analysis.
Drop duplicates, blank categories, `price`/`amount`/`qty` <= 0, invalid emails,
and tidy the messy `status` / `sub_tier` text.

In [4]:
# products: drop blank category + price<=0, one row per product_id
products["category"] = products["category"].astype("string").str.strip()
products = products[products["category"].notna() & (products["category"] != "") & (products["price"] > 0)]
products = products.drop_duplicates(subset=["product_id"])

# order_items: drop dups + qty/unit_price<=0, recompute subtotal
order_items = order_items.drop_duplicates().query("qty > 0 and unit_price > 0").copy()
order_items["subtotal"] = (order_items["qty"] * order_items["unit_price"]).round(2)

# orders: one row per order_id, amount>0, tidy status
orders = orders.drop_duplicates().drop_duplicates(subset=["order_id"]).copy()
orders["status"] = orders["status"].str.strip().str.lower().replace({"canceled": "cancelled"})
orders = orders.query("amount > 0")

# customers: one row per id, valid email, Title-case tier
customers = customers.drop_duplicates().drop_duplicates(subset=["customer_id"]).copy()
customers = customers[customers["email"].astype("string").str.contains(r"@.+\.", regex=True, na=False)]
customers["sub_tier"] = customers["sub_tier"].str.strip().str.title()

In [5]:
print(f"clean -> products={len(products)}, order_items={len(order_items)}, "
      f"orders={len(orders)}, customers={len(customers)}")
assert len(products) == 50, "expected 50 clean products"
assert set(products["category"]) == {"Beverage", "Snack", "Personal Care", "Household", "Food"}

clean -> products=50, order_items=9122, orders=2966, customers=984


### TASK 3 — Build one enriched **line-item** table.

Join `order_items` → `products` (category, brand) → `orders` (status, customer)
→ `customers` (tier). This denormalised table is what we analyse.

In [6]:
sales = (
    order_items
    .merge(products[["product_id", "product_name", "category", "brand"]], on="product_id", how="inner")
    .merge(orders[["order_id", "customer_id", "order_date", "status"]], on="order_id", how="inner")
    .merge(customers[["customer_id", "sub_tier"]], on="customer_id", how="left")
)
sales["month"] = sales["order_date"].dt.strftime("%Y-%m")

In [7]:
print("sales line items:", sales.shape)
assert {"category", "brand", "status", "sub_tier", "subtotal", "month"} <= set(sales.columns)
sales.head()

sales line items: (9024, 14)


,order_item_id,order_id,product_id,qty,unit_price,subtotal,product_name,category,brand,customer_id,order_date,status,sub_tier,month
0,5405,4135,3050,4,237.22,948.88,Tao Kae Noi Product 3050,Snack,Tao Kae Noi,1391,2022-03-22,completed,Gold,2022-03
1,12046,6314,3012,7,253.48,1774.36,Unilever Product 3012,Food,Unilever,1599,2022-07-03,completed,Platinum,2022-07
2,7996,4974,3049,4,45.80,183.20,Mama Product 3049,Food,Mama,1859,2023-04-30,processing,Platinum,2023-04
3,11380,6091,3048,5,154.34,771.70,PepsiCo Product 3048,Food,PepsiCo,1644,2023-10-14,completed,Silver,2023-10
4,13279,6736,3027,8,263.71,2109.68,Tao Kae Noi Product 3027,Food,Tao Kae Noi,1368,2022-06-06,processing,Silver,2022-06


### TASK 4 — Total revenue per category, high → low.

In [8]:
by_category = sales.groupby("category")["subtotal"].sum().sort_values(ascending=False)
by_category.round(0)

category
Food             6051788.0
Snack            2765586.0
Personal Care    2312378.0
Beverage         2270435.0
Household        1365577.0
Name: subtotal, dtype: float64

### TASK 5 — Top 5 products by total revenue.

In [9]:
sales.groupby("product_name")["subtotal"].sum().sort_values(ascending=False).head(5).round(0)

product_name
Oishi Product 3006     958852.0
Mama Product 3049      946478.0
Singha Product 3038    928488.0
Oishi Product 3004     773383.0
Nestle Product 3041    731015.0
Name: subtotal, dtype: float64

### TASK 6 — Pivot: revenue by category (rows) × status (columns).

In [10]:
pivot = sales.pivot_table(values="subtotal", index="category", columns="status",
                          aggfunc="sum", fill_value=0)
pivot.astype(int)

status,cancelled,completed,processing
category,,,
Beverage,238917,879319,1152197
Food,1124824,3538537,1388426
Household,182522,942066,240987
Personal Care,276909,1144872,890596
Snack,575117,1884868,305600


### TASK 7 — Revenue by customer tier (Bronze → Platinum).

In [11]:
sales.groupby("sub_tier")["subtotal"].sum().sort_values(ascending=False).round(0)

sub_tier
Silver      5254038.0
Platinum    3591842.0
Gold        3203622.0
Bronze      2551965.0
Name: subtotal, dtype: float64

### BONUS — Revenue per brand.

In [12]:
sales.groupby("brand")["subtotal"].sum().sort_values(ascending=False).round(0)

brand
Oishi          2609386.0
Tao Kae Noi    2396984.0
Colgate        2296533.0
Mama           1946444.0
Nestle         1486518.0
Singha         1364948.0
PepsiCo         975595.0
Lays            906444.0
Unilever        567836.0
CP              215076.0
Name: subtotal, dtype: float64

**Done!** The top category should be **Food**. Try changing a cleaning
rule (e.g. keep `amount <= 0`) and predict how the totals move.